## Creacion de las tablas de la capa recursos 
- Crear los schemas y definir los tipos de datos para las tablas de la capa recursos.

## Importando Modulos 

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

ruta_actual = Path.cwd()
sys.path.append(str(ruta_actual.parent / "01-funciones-compartidas"))

from funciones_conexion import obtener_ruta

## recursos.customers

### Resumen Columnas
|       Columna      |                        Descripción                        |
|--------------------|-----------------------------------------------------------|
| `customer_id`      | Identificador unico del cliente (`C000001`…)              |
| `full_name`        | Nombre completo del cliente (NOMBRE APELLIDO1 APELLIDO2)  |
| `email`            | Email de contacto del cliente                             |
| `phone`            | Número de contacto del cliente                            |
| `country`          | País de origen del cliente                                |
| `city`             | Ciudad de origen del cliente                              |
| `birth_date`       | Fecha de nacimiento del cliente                           |
| `gender`           | Género del cliente (M/F/X)                                |
| `segment`          | Segmeto de cliente`standard`, `premium` o `vip`           |
| `marketing_opt_in` | Consentimiento para recibir marketing                     |
| `registered_at`    | Fecha de registro del cliente                             |
| `updated_at`       | Fecha de actualización de la información del cliente      |
| `source_system`    | Sistema origen: `crm_import`, `web`, `app`, `call_center` |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dlh_nova_shop.recursos.customers(
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    country STRING,
    city STRING,
    birth_date STRING,
    gender STRING,
    segment STRING,
    marketing_opt_in STRING,
    registered_at STRING,
    updated_at STRING,
    source_system STRING,
    dlh_fecha_ingesta TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA 
-- LOCATION obtener_ruta("recursos") + "/customers"
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
);

## recursos.products

## Resumen Columnas
|       Campo       |                        Descripción                        |
|-------------------|-----------------------------------------------------------|
| `product_id`      | Identificación del producto                               |
| `name`            | Nombre del Producto                                       |
| `brand`           | Marca del producto                                        |
| `category`        | Struct `{l1, l2}`: categoría y subcategoría               |
| `price`           | Struct `{base_usd, currency}`: precio de lista en USD     |
| `attributes`      | Struct con atributos **opcionales** que cambian según la categoría (color, peso, talla, garantía…) |
| `tags`            | Array de etiquetas                                        |
| `suppliers`       | Array de structs `{supplier_id, cost_usd, lead_time_days, country_origin, is_preferred}` |
| `active`          | Valor booleano que define si el producto es activo o inactivo |
| `created_at`      | Fecha de creacion de ese registro (fecha como texto ISO-8601) |
| `discontinued_at` | Fecha de descontinuacion          (fecha como texto ISO-8601) |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dlh_nova_shop.recursos.products(
    product_id STRING,
    name STRING,
    brand STRING,
    category STRUCT<
        l1: STRING, 
        l2: STRING
    >,
    price STRUCT<
        base_usd: DECIMAL(10,2), 
        currency: STRING
    >,
    attributes STRUCT<
        battery_required: BOOLEAN, 
        color: STRING, 
        connectivity: STRING, 
        material: STRING, 
        min_age_years: INTEGER, 
        organic: BOOLEAN, 
        shelf_life_days: INTEGER, 
        size: STRING, 
        vegan: BOOLEAN, 
        volume_ml: INTEGER,
        warranty_months: INTEGER,
        weight_kg: DECIMAL(10,2)
    >, 
    tags ARRAY<STRING>,
    suppliers ARRAY<
        STRUCT<
            cost_usd: DECIMAL(10,2),
            country_origin: STRING, 
            is_preferred: BOOLEAN, 
            lead_time_days: INTEGER, 
            supplier_id: STRING
        >
    >, 
    active BOOLEAN,
    created_at STRING,
    discontinued_at STRING,
    dlh_fecha_ingesta TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA 
-- LOCATION obtener_ruta("recursos") + "/products"
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
);

## recursos.sales_lines


## Resumen Columnas
| Columna | Descripción |
|---|---|
| `order_id`            | Identificador unico de la venta                                            |
| `line_number`         | Número de línea, es decir puede existir una venta con 3 productos de línea |
| `order_ts`            | Momento de la compra (UTC)                                                 |
| `customer_id`         | Clave foránea de cliente                                                   |
| `product_id`          | Clave foránea del producto                                                 |
| `quantity`            | Cantidad de productos                                                      |
| `unit_price_local`    | Precio unitario en la modena local                                         |
| `currency`            | Moneda local                                                               |
| `discount_pct`        | Porcentaje de descuento                                                    |
| `coupon_code`         | Cupón usado                                                                |
| `status`              | Estado **de la línea**: completed, cancelled, returned, pending            |
| `payment_method`      | Método de pago                                                             |
| `channel`             | Canal de venta                                                             |
| `country`             | País de la venta                                                           |
| `shipping_cost_local` | Costo de envío **del pedido** (se repite en cada línea del pedido)         |
| `delivered_at`        | Fecha de entrega al cliente (UTC; nulo si no entregado)                    |
| `ingested_at`         | Cuándo llegó la fila a nuestro lago de datos                               |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dlh_nova_shop.recursos.sales_lines (
    order_id STRING,
    line_number INTEGER,
    order_ts STRING,
    customer_id STRING,
    product_id STRING,
    quantity INTEGER,
    unit_price_local DOUBLE,
    currency STRING,
    discount_pct DOUBLE,
    coupon_code STRING,
    status STRING,
    payment_method STRING,
    channel STRING,
    country STRING,
    shipping_cost_local DOUBLE,
    delivered_at STRING,
    ingested_at STRING,
    dlh_fecha_ingesta TIMESTAMP DEFAULT current_timestamp()
)
USING DELTA
-- LOCATION obtener_ruta("recursos") + "/products"
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
);

## recursos.web_events

## Resumen Columnas
| Campo | Descripción |
|---|---|
| `anonymous_id`| Cookie/dispositivo |
| `customer_id` | `customer_id` solo existe si hay sesión iniciada |
| `device` | struct `{browser, os, type, ua}` (`ua` solo aparece cuando el SDK no reconoce el agente) |
| `event_id` | Identificador del evento (los reintentos del SDK lo repiten) |
| `event_ts`| Cuándo ocurrió en el dispositivo y cuándo llegó al servidor (texto ISO-8601, UTC) |
| `event_type` | `page_view`, `search`, `product_view`, `add_to_cart`, `begin_checkout`, `purchase` |
| `geo` | Struct `{country}` |
| `page` | Struct `{referrer, url}` |
| `platform` | `web`/`ios`/`android` |
| `props` | Struct según el evento: `items`,`order_id`, `product_id`, `quantity`, `results`,`search_term`|
| `received_at` | Cuándo ocurrió en el dispositivo y cuándo llegó al servidor (texto ISO-8601, UTC) |
| `utm` | Struct `{campaign, medium, source }`; solo viene en el **primer evento** de una visita con campaña |

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dlh_nova_shop.recursos.web_events(
    anonymous_id STRING,
    customer_id STRING,
    device STRUCT<
        browser STRING, 
        os STRING, 
        type STRING, 
        ua STRING
    >,
    event_id STRING,
    event_ts STRING,
    event_type STRING,
    geo STRUCT<
        country STRING
    >,
    page STRUCT<
        referrer STRING, 
        url STRING
    >,
    platform STRING,
    props STRUCT<
        items INTEGER,
        order_id STRING,
        product_id STRING,
        quantity INTEGER,
        results INTEGER,
        search_term STRING
    >,
    received_at STRING,
    utm STRUCT<
        campaign STRING, 
        medium STRING, 
        source STRING
    >,
    dlh_fecha_ingesta TIMESTAMP DEFAULT current_timestamp()
)
-- LOCATION obtener_ruta("recursos") + "/products"
TBLPROPERTIES (
    'delta.feature.allowColumnDefaults' = 'supported'
);